# Redrob Semantic Ranker — Phase 2

Combines, for each candidate:
- **structured fit** (`struct_score` from explore.ipynb: ML-at-product, retrieval/eval evidence, verified IR assessment, location, experience band)
- **semantic fit** = cosine(JD, candidate **career prose**) via `bge-small` — catches the no-buzzword Tier-5 and the meaning the phrase-lists miss
- **behavioral** availability multiplier (JD directive #3)

`final = (0.65 * struct_norm + 0.35 * sem_norm) * behavioral`, honeypots excluded.
Candidate and JD embeddings are precomputed once and cached (the 5-min live ranker just loads arrays).

In [1]:
import json, os, time
import numpy as np, pandas as pd

DATA='data/candidates.jsonl'
MAX_SEQ_LENGTH = 256
EMB=f'artifacts/cand_emb_{MAX_SEQ_LENGTH}.npy'
IDS='artifacts/cand_ids.json'
JD_EMB=f'artifacts/jd_emb_{MAX_SEQ_LENGTH}.npy'

feat=pd.read_parquet('artifacts/features.parquet').set_index('candidate_id')
print('features:', feat.shape, '| honeypots:', int(feat.honeypot.sum()))

# JD distilled to the fit-theory, used as the retrieval query (bge wants the query prefix).
# Cache this too so the final ranking step can run with no network/model call.
JD_QUERY=('Represent this sentence for searching relevant passages: '
  'Senior AI/ML engineer who has built and shipped ranking, retrieval, search, recommendation and '
  'matching systems to real users in production at product companies. Embeddings-based retrieval, '
  'vector search, hybrid search, learning-to-rank, semantic search; rigorous evaluation with NDCG, MRR, '
  'MAP, A/B testing and offline-online correlation. Strong Python. 5-9 years experience, applied ML at '
  'product companies rather than pure services. Based in or willing to relocate to Pune or Noida, India.')

model = None
if os.path.exists(JD_EMB):
    jd_emb=np.load(JD_EMB)
    print('loaded cached JD embedding:', jd_emb.shape)
else:
    import torch
    from sentence_transformers import SentenceTransformer
    # Use Apple MPS / CUDA only for one-time precomputation; cached ranking is CPU/no-network.
    DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
    model = SentenceTransformer('BAAI/bge-small-en-v1.5', device=DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH
    print('device:', model.device, '| max_seq_length:', model.max_seq_length)
    jd_emb=model.encode(JD_QUERY, normalize_embeddings=True)
    np.save(JD_EMB, jd_emb)
    print('cached JD embedding:', jd_emb.shape)


features: (100000, 19) | honeypots: 80
loaded cached JD embedding: (384,)


In [2]:
# Build candidate 'work text' = summary + job titles + job descriptions (NOT the noise skill list),
# encode with bge-small, and CACHE. Re-runs load the cache instantly.
# Cache key includes max_seq_length so changing it forces a clean re-encode.
if os.path.exists(EMB) and os.path.exists(IDS):
    cand_ids=json.load(open(IDS)); cand_emb=np.load(EMB)
    print('loaded cached embeddings:', cand_emb.shape)
else:
    if model is None:
        import torch
        from sentence_transformers import SentenceTransformer
        DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
        model = SentenceTransformer('BAAI/bge-small-en-v1.5', device=DEVICE)
        model.max_seq_length = MAX_SEQ_LENGTH
        print('device:', model.device, '| max_seq_length:', model.max_seq_length)
    cand_ids=[]; texts=[]
    with open(DATA) as f:
        for line in f:
            line=line.strip()
            if not line: continue
            c=json.loads(line); p=c.get('profile',{}) or {}
            parts=[p.get('summary') or '']
            for j in (c.get('career_history',[]) or []):
                parts.append(j.get('title') or ''); parts.append(j.get('description') or '')
            cand_ids.append(c['candidate_id']); texts.append('. '.join(t for t in parts if t))
    bs = 256 if model.device.type in ('mps','cuda') else 64   # smaller batches are faster on CPU
    print(f'encoding {len(texts)} candidates on {model.device} (one-time)...')
    t=time.time()
    cand_emb=model.encode(texts, normalize_embeddings=True, batch_size=bs, show_progress_bar=True)
    print('encoded in', round(time.time()-t), 's')
    np.save(EMB, cand_emb); json.dump(cand_ids, open(IDS,'w'))
    print('cached:', cand_emb.shape)


loaded cached embeddings: (100000, 384)


In [3]:
# Semantic similarity to JD, then the hybrid blend.
# We encoded the FULL pool (needed later for semantic-incoherence hunting of undetected honeypots),
# but we DROP the known honeypots up front so they can't contaminate normalization or the ranking.
# NB: column is 'sem_sim' not 'sem' -- 'sem' collides with the pandas DataFrame.sem() method.
sem_vec = cand_emb @ jd_emb                  # cosine (both normalized)
sem_df = pd.DataFrame({'candidate_id':cand_ids,'sem_sim':sem_vec}).set_index('candidate_id')

R = feat[~feat.honeypot].join(sem_df)        # genuine pool only (known honeypots excluded here)
print('ranking pool (honeypots removed):', len(R), 'of', len(feat))
R['struct_norm'] = (R['struct_score'] / R['struct_score'].max()).clip(0,1)
lo,hi = R['sem_sim'].quantile(0.01), R['sem_sim'].quantile(0.99)
R['sem_norm'] = ((R['sem_sim'] - lo)/(hi-lo)).clip(0,1)
R['fit']   = 0.65*R['struct_norm'] + 0.35*R['sem_norm']  # structured gates domain, semantic fine-ranks
R['final'] = R['fit'] * R['behavioral']                 # availability multiplier

top = R.sort_values(['final','candidate_id'], ascending=[False,True]).head(100)
print('TOP-100 tier distribution:', top['tier'].value_counts().sort_index().to_dict())
print('final score range in top-100:', round(top['final'].min(),3),'..',round(top['final'].max(),3))
cols=['tier','struct_score','sem_sim','fit','behavioral','final','current_title','current_industry','yoe','india']
top[cols].head(20)

ranking pool (honeypots removed): 99920 of 100000


TOP-100 tier distribution: {5: 100}
final score range in top-100: 0.72 .. 0.889


,tier,struct_score,sem_sim,fit,behavioral,final,current_title,current_industry,yoe,india
candidate_id,,,,,,,,,,
CAND_0077337,5,6.600,0.864101,0.929730,0.956,0.888822,staff machine learning engineer,fintech,7.0,True
CAND_0006418,5,7.000,0.831187,0.964865,0.917,0.884781,machine learning engineer,conversational ai,5.7,True
CAND_0081846,5,7.000,0.829821,0.964865,0.914,0.881886,lead ai engineer,fintech,6.7,True
CAND_0000031,5,6.500,0.886769,0.920946,0.945,0.870294,recommendation systems engineer,food delivery,6.0,True
CAND_0079387,5,6.550,0.853997,0.925338,0.932,0.862415,ai engineer,software,6.9,True
CAND_0065195,5,6.550,0.857853,0.925338,0.929,0.859639,search engineer,fintech,5.1,True
CAND_0006567,5,7.000,0.784910,0.953976,0.901,0.859533,senior ai engineer,internet,7.9,True
CAND_0061257,5,6.300,0.805402,0.903378,0.943,0.851886,staff machine learning engineer,internet,8.0,True
CAND_0091909,5,7.400,0.860314,1.000000,0.849,0.849000,machine learning engineer,ai/ml,6.9,True


## Audit the top-100 for honeypots

The plan: rank first, then scrutinize only the shortlist. Re-run the full impossibility battery on the
top-100 raw records (should be 0, since honeypots are excluded), and surface them for manual inspection
of any semantic-incoherence honeypots the rules can't catch.

In [4]:
top_ids=set(top.index)
recs={}
with open(DATA) as f:
    for line in f:
        line=line.strip()
        if not line: continue
        c=json.loads(line)
        if c['candidate_id'] in top_ids: recs[c['candidate_id']]=c
known_hp = top.index.isin(feat[feat.honeypot].index)
print('known honeypots in top-100:', int(known_hp.sum()), '(should be 0)')
print()
print('Top-15 shortlist (for eyeballing fit + any incoherence):')
order=list(top.index)
for i,cid in enumerate(order[:15], start=1):
    p=recs[cid]['profile']; r=top.loc[cid]
    print()
    print(f"#{i} {cid} | {p.get('current_title')} @ {p.get('current_company')} ({p.get('current_industry')}) | yoe={p.get('years_of_experience')} | final={r['final']:.3f} sem={r['sem_sim']:.3f} tier={int(r['tier'])}")
    print('   ', (p.get('summary') or '')[:240])

known honeypots in top-100: 0 (should be 0)

Top-15 shortlist (for eyeballing fit + any incoherence):

#1 CAND_0077337 | Staff Machine Learning Engineer @ Paytm (Fintech) | yoe=7.0 | final=0.889 sem=0.864 tier=5
    Senior AI engineer with 7.0 years of hands-on experience building production ML systems, with a focus on search, retrieval, and ranking. Most recently, I designed the company's first hybrid retrieval system combining BM25 with dense vector 

#2 CAND_0006418 | Machine Learning Engineer @ Verloop.io (Conversational AI) | yoe=5.7 | final=0.885 sem=0.831 tier=5
    Machine learning engineer with 5.7 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment

#3 CAND_0081846 | Lead AI Engineer @ Razorpay (Fintech) | yoe=6.7 | final=0.882 sem=0.830 tier=5
    Senior AI engineer with 6.7 years of hands-on experience building production 

In [5]:
# ===== EXPORT the top-100 (rank order) with scores + full records for inspection =====
import csv, datetime as dt, re

TODAY = dt.date(2026, 6, 29)
def parse_date(s):
    try: return dt.date.fromisoformat(s)
    except Exception: return None

def evidence_terms(rec):
    text = ' '.join([
        rec.get('profile',{}).get('summary',''),
        ' '.join((j.get('title','')+' '+j.get('description','')) for j in rec.get('career_history',[]))
    ]).lower()
    term_map = [
        ('hybrid retrieval','hybrid retrieval'), ('semantic search','semantic search'),
        ('candidate-jd','candidate-JD matching'), ('recommendation system','recommendation systems'),
        ('ranking model','ranking models'), ('ranking system','ranking systems'),
        ('learning-to-rank','learning-to-rank'), ('bm25','BM25'), ('faiss','FAISS'),
        ('qdrant','Qdrant'), ('milvus','Milvus'), ('weaviate','Weaviate'), ('pinecone','Pinecone'),
        ('elasticsearch','Elasticsearch'), ('vector','vector search'), ('embedding','embedding-based retrieval'),
        ('ndcg','NDCG'), ('mrr','MRR'), ('a/b test','A/B testing'), ('offline','offline evaluation'),
        ('production','production deployment'), ('serving','serving real users'), ('latency','latency work'),
    ]
    out=[]
    for needle,label in term_map:
        if needle in text and label not in out:
            out.append(label)
    return out[:5]

def make_reasoning(rec, row):
    p=rec.get('profile',{}) or {}; sig=rec.get('redrob_signals',{}) or {}
    terms=evidence_terms(rec)
    term_txt=', '.join(terms[:3]) if terms else 'applied ML/product engineering evidence'
    la=parse_date(sig.get('last_active_date'))
    inactive=(TODAY-la).days if la else None
    resp=sig.get('recruiter_response_rate')
    notice=sig.get('notice_period_days')
    loc=p.get('location') or p.get('country') or 'location unspecified'
    first=(f"{p.get('years_of_experience')} years as {p.get('current_title')} in {p.get('current_industry')}, "
           f"with profile/career evidence for {term_txt}.")
    bits=[]
    if resp is not None: bits.append(f"{round(resp*100)}% recruiter response")
    if inactive is not None: bits.append(f"active {inactive} days ago")
    if notice is not None: bits.append(f"{notice}-day notice")
    bits.append(loc)
    second='Behavioral/logistics signals: ' + ', '.join(bits) + '.'
    return first + ' ' + second

order = list(top.index)
out = []
submission_rows=[]
for rank, cid in enumerate(order, start=1):
    r = top.loc[cid]
    rec = recs.get(cid, {})
    reasoning = make_reasoning(rec, r)
    out.append({
        'rank': rank,
        'candidate_id': cid,
        'scores': {
            'final':        round(float(r['final']), 4),
            'sem':          round(float(r['sem_sim']), 4),
            'struct_score': round(float(r['struct_score']), 3),
            'fit':          round(float(r['fit']), 4),
            'behavioral':   round(float(r['behavioral']), 3),
            'tier':         int(r['tier']),
        },
        'reasoning':      reasoning,
        'profile':        rec.get('profile', {}),
        'career_history': rec.get('career_history', []),
        'education':      rec.get('education', []),
        'skills':         rec.get('skills', []),
        'redrob_signals': rec.get('redrob_signals', {}),
    })
    submission_rows.append({
        'candidate_id': cid,
        'rank': rank,
        'score': round(float(r['final']), 6),
        'reasoning': reasoning,
    })
with open('artifacts/top100.json', 'w') as fo:
    json.dump(out, fo, indent=2)
with open('artifacts/submission.csv', 'w', newline='') as fo:
    writer=csv.DictWriter(fo, fieldnames=['candidate_id','rank','score','reasoning'])
    writer.writeheader(); writer.writerows(submission_rows)
print('wrote artifacts/top100.json + artifacts/submission.csv with', len(out), 'candidates (rank order)')


wrote artifacts/top100.json + artifacts/submission.csv with 100 candidates (rank order)
